In [20]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
import pandas as pd
import requests
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("AIMLAPI_KEY","")
assert API_KEY, "ERROR: AIMLAPI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )

model = 'text-embedding-ada-002'

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "embedding_index_3m.json"

In [22]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Завантаження індексу відеосесій
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

In [23]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # створення копії набору даних
    video_vectors = dataset.copy()

    # отримання вбудовувань для запиту за допомогою прямого виклику API
    response = requests.post(
        "https://api.aimlapi.com/v1/embeddings",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "input": query,
            "encoding_format": "float"
        }
    )
    
    data = response.json()

    print(data)

    query_embeddings = data["data"][0]["embedding"]
    
    # створення нового стовпчика з обчисленою подібністю для кожного рядка
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # повернення верхніх рядків
    return video_vectors.head(rows)

In [24]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """конвертація часу з формату 00:00:00 в секунди"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nВідео, схожі на '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Резюме: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Схожість: {row['similarity']}")
        print(f"   Доповідачі: {row['speaker']}")

In [25]:
pd_vectors = load_dataset(DATASET_NAME)

# отримання запиту користувача з вводу
while True:
    query = input("Введіть запит: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

{'requestId': 'HcMuQg8rRI9E0zrqXGlNM', 'statusCode': 401, 'timestamp': '2025-12-03T22:40:49.342Z', 'path': '/v1/embeddings', 'message': 'Unauthorized'}


KeyError: 'data'

___

## Виконати пошук фрагментів, що містять "Python". Порахувати загальну кількість згадувань цього слова.